In [2]:
import sys
sys.path.append('../Src')

from platform import python_version
import numpy as np
import pandas as pd
from tqdm import tqdm
# user functions
import utility
import db_connection

import importlib

import glob
#  The glob library (or more accurately, the glob module in Python) is a built-in Python module that provides a way to find all 
# the pathnames matching a specified pattern according to the rules used by the Unix shell.
# It's extremely useful when you need to discover files or directories based on wildcard patterns, without having to manually 
# iterate through directories and check filenames.
import langchain
from langchain.document_loaders import DirectoryLoader
from langchain_community.document_loaders.unstructured import UnstructuredFileLoader
from langchain.document_loaders import UnstructuredPDFLoader  # or langchain_community if using that

# initializing variables
RANDOM_STATE = 1776

# print versions
print("LangChain Version: " + langchain.__version__)
print("Numpy Version: " + np.__version__)
print("Pandas Version: " + pd.__version__)
# print("Seaborn Version: " + sns.__version__)
# print("Matplotlib Version: " + plt.matplotlib.__version__)
print("Python Version: " + python_version())

LangChain Version: 0.3.27
Numpy Version: 2.2.6
Pandas Version: 2.3.2
Python Version: 3.13.7


#### This line of code initializes a DirectoryLoader object from LangChain, configured to find and prepare specific files for loading. Let's break down its components:
- from langchain.document_loaders import DirectoryLoader:
    - This line imports the DirectoryLoader class from LangChain's document_loaders module. This class is designed to load multiple documents from a specified directory.
        - loader = DirectoryLoader(...):
    - This creates an instance of the DirectoryLoader class and assigns it to the variable loader. This loader object is now configured but hasn't actually loaded any documents yet.
        - './data':
    - This is the first argument to DirectoryLoader and specifies the path to the directory you want to load documents from.
        - ./data means a directory named data located in the current working directory where your Python script is being executed.
        - glob="**/*.pdf":
        - This is a crucial argument that defines a glob pattern to filter which files within the specified directory should be loaded.
        - **: This is a wildcard that matches any directory (including subdirectories) zero or more levels deep. So, it tells the loader to search recursively within ./data.
        - *.pdf: This wildcard matches any file ending with .pdf.
        - Combined (**/*.pdf): This pattern instructs the DirectoryLoader to find all files ending with .pdf within the ./data directory and any of its subdirectories.



In [40]:
importlib.reload(db_connection)

<module 'db_connection' from '/Users/sir/Desktop/Project/RAG/Notebook/../Src/db_connection.py'>

### For standalone testing for PostgreSQL connection

In [38]:
import asyncio

# For standalone testing
db = db_connection.AsyncPostgresConnector()

# 1. Test connection
await db.test_connection()

# 2. Gracefully close connection
await db.close_connection()

🔄 Initializing async SQLAlchemy engine...
✅ Async SQLAlchemy engine initialized.
✅ Session factory initialized.
✅ Connection successful. DataFrame result:
      oid  extname  extowner  extnamespace  extrelocatable extversion extconfig extcondition
0   14009  plpgsql        10            11           False        1.0      None         None
1   24576   vector        10          2200            True      0.8.1      None         None
2  156141  pg_trgm        10          2200            True        1.6      None         None
🛑 Shutting down and gracefully disposing of async engine...
🗑️ All pooled connections closed.


In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf")
docs = loader.load_and_split()

In [ ]:
type(docs), len(docs)

In [2]:
# from docx import Document

# doc = Document("your_file.docx")
# for para in doc.paragraphs:
#     print(para.text)



# from langchain_community.document_loaders.excel import UnstructuredExcelLoader

# # Load the Excel file (choose mode: "single" or "elements")
# loader = UnstructuredExcelLoader("your_file.xlsx", mode="elements")
# docs = loader.load()
# # List All Sheet Names
# for doc in docs:
#     print(doc.metadata.get("sheet_name"), doc.page_content[:100])

# Use UnstructuredExcelLoader to bring Excel data into LangChain. You can load entire files or 
# individual sheets, and each document will include both text and metadata for downstream processing

### Modes Explained
- "single" mode (default):
- Loads the entire Excel file as one document.
- The document's metadata includes an HTML representation of the table under the text_as_html key.

### "elements" mode:
- Loads each sheet as a separate table element.
- Each element is a separate document, and its metadata contains details about the sheet and an HTML version of the table.
- What You Get
    - Each loaded document has:
    - page_content: The raw text extracted from the Excel file or sheet.
    - metadata: Information about the sheet, file, and (in "single" mode) an HTML version of the table.

### Typical Workflow
- Use UnstructuredExcelLoader to load Excel data into LangChain Document objects.
- Process, chunk, embed, or retrieve from these documents as needed for your LLM or RAG pipeline.
- Example: List All Sheet Names
    - python
        - for doc in docs:
        - print(doc.metadata.get("sheet_name"), doc.page_content[:100])
### Summary Table
- Loader	File Types Supported	Modes	Special Features
- UnstructuredExcelLoader	.xlsx, .xls	"single", "elements"	HTML table in metadata, sheet-level docs
### In summary:
- Use UnstructuredExcelLoader to bring Excel data into LangChain. You can load entire files or individual sheets, and each document will include both text and metadata for downstream processing

## User Function(s)

In [3]:
import fitz
import glob
import math

def get_max_longest_edge_for_pdfs(pdf_paths):
    max_edge = 0
    for pdf_path in pdf_paths:
        doc = fitz.open(pdf_path)
        for page in doc:
            rect = page.rect
            max_edge = max(max_edge, rect.width, rect.height)
        doc.close()
    return max_edge

# List your PDF files
pdf_files = glob.glob('../Docs/**/*.pdf', recursive=True)

# Compute max longest edge
max_longest_edge = get_max_longest_edge_for_pdfs(pdf_files)

rounded_edge = math.ceil(max_longest_edge)
print(rounded_edge) 


843


## Unstructured File Loader

In [ ]:
loader = DirectoryLoader(
    '../Docs',
    glob="**/*.pdf",
    loader_cls=UnstructuredFileLoader, #  (default, works with many file types including PDFs, HTML, Markdown)
    loader_kwargs={"languages": ["eng"],
                   "size": {"longest_edge": rounded_edge}, # Set a fixed size from function above
                   "mode": "elements", 
                   "strategy": "fast",
                   "infer_table_structure": True,
                   "extract_images_in_pdf": False,
                   "show_progress": True,
                   },
    use_multithreading=True,
    max_concurrency=8,
    show_progress=True,
    silent_errors=True,         # skip files that raise errors
    load_hidden=False,          # don't load hidden files/folders
    # recursive=True,           # load files from subfolders recursively
    sample_size=1,             # max 100 files, useful for testing
    randomize_sample=True,      # shuffle files before loading
    sample_seed=1776            # reproducible shuffle for sampling
)

# Actually load the documents
documents = loader.load()

print(f"Total documents loaded: {len(documents)}\n")
# print(documents[0].metadata['source'])  # Display the source of the first document
# print(documents[0])  # Display the content of the first document

direct_file = documents.copy()


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:07<00:00,  7.26s/it]

Total documents loaded: 177



(list, 11)

In [80]:
docs[0].metadata

{'producer': 'OpenPDF 1.0.0-SNAPSHOT; modified using iTextSharp 5.4.1 ©2000-2012 1T3XT BVBA (AGPL-version); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV',
 'creator': '',
 'creationdate': '2025-04-21T05:16:08+00:00',
 'source': '../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf',
 'file_path': '../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf',
 'total_pages': 6,
 'format': 'PDF 1.4',
 'title': 'Multi-Tiered RAG-Based Chatbot for Mental Health Support',
 'author': '',
 'subject': '2025 Eighth International Women in Data Science Conference at Prince Sultan University (WiDS PSU);2025; ; ;10.1109/WiDS-PSU64963.2025.00041',
 'keywords': '',
 'moddate': '2025-05-27T15:26:20-04:00',
 'trapped': '',
 'modDate': "D:20250527152620-04'00'",
 'creationDate': 'D:20250421051608Z',
 'page': 0}

In [92]:
docs[0].metadata.get("source","N/A")

'../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf'

In [93]:
docs[0].metadata.get("producer","N/A")

'OpenPDF 1.0.0-SNAPSHOT; modified using iTextSharp 5.4.1 ©2000-2012 1T3XT BVBA (AGPL-version); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV'

In [97]:
from datetime import datetime

datetime.fromisoformat(docs[0].metadata.get("creationdate","N/A"))

datetime.datetime(2025, 4, 21, 5, 16, 8, tzinfo=datetime.timezone.utc)

In [26]:
from langchain_huggingface import HuggingFaceEmbeddings

# embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2") # 384 Dimensions
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2") # 768 Dimensions
vectors = [embedding_model.embed_query(doc.page_content) for doc in docs]

In [ ]:



importlib.reload(utility)  # Reload the sys module to ensure any changes are applied

x = utility.paper_id_from_path("../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf")


In [5]:
all_keys = set()
for doc in direct_file:
    all_keys.update(doc.metadata.keys())
print(f"All unique metadata keys: {all_keys}")

All unique metadata keys: {'filetype', 'source', 'last_modified', 'coordinates', 'languages', 'filename', 'element_id', 'page_number', 'file_directory', 'category', 'parent_id'}


In [ ]:
# list comprehension that iterates over every document object
loaded_sources = [doc.metadata.get("source", "") for doc in direct_file]
loaded_files = set(loaded_sources)

loaded_files

{'../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf'}

In [13]:
len(direct_file), type(direct_file)

(177, list)

In [8]:
direct_file[0:5]

[Document(metadata={'source': '../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf', 'coordinates': {'points': ((11.0, 112.10400000000016), (11.0, 241.24000000000024), (19.0, 241.24000000000024), (19.0, 112.10400000000016)), 'system': 'PixelSpace', 'layout_width': 612.0, 'layout_height': 792.0}, 'file_directory': '../Docs', 'filename': 'Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf', 'languages': ['eng'], 'last_modified': '2025-09-18T16:01:17', 'page_number': 1, 'filetype': 'application/pdf', 'category': 'UncategorizedText', 'element_id': 'a99371efa682e4b6106da71412ede5a3'}, page_content='1 4 0 0 0 5 2 0 2 3 6 9 4 6 U S P - S D W / 9 0 1 1 0 1 : I'),
 Document(metadata={'source': '../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf', 'coordinates': {'points': ((11.0, 132.34400000000016), (11.0, 134.36000000000013), (19.0, 134.36000000000013), (19.0, 132.34400000000016)), 'system': 'PixelSpace', 'layout_width': 612.0, 'layout_height': 792

In [84]:
direct_file[0].metadata

{'source': '../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf',
 'coordinates': {'points': ((11.0, 112.10400000000016),
   (11.0, 241.24000000000024),
   (19.0, 241.24000000000024),
   (19.0, 112.10400000000016)),
  'system': 'PixelSpace',
  'layout_width': 612.0,
  'layout_height': 792.0},
 'file_directory': '../Docs',
 'filename': 'Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf',
 'languages': ['eng'],
 'last_modified': '2025-09-18T16:01:17',
 'page_number': 1,
 'filetype': 'application/pdf',
 'category': 'UncategorizedText',
 'element_id': 'a99371efa682e4b6106da71412ede5a3'}

In [ ]:
direct_file[0].metadata.get("source","")

'../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf'

In [14]:
# Assuming 'documents' is the list of Document objects you loaded
headings = []

for doc in documents:
    # Get the category from the document's metadata
    category = doc.metadata.get('category')
    
    # Check if the category is a heading or title
    if category in ['Header', 'Title']:
        headings.append(doc)

print(f"Total headings found: {len(headings)}\n")

# Print the content of each found heading
for heading in headings:
    print(f"Category: {heading.metadata.get('category')}")
    print(f"Content: {heading.page_content}")
    print("-" * 50)

Total headings found: 63

Category: Title
Content: I
--------------------------------------------------
Category: Title
Content: O D | E E E I
--------------------------------------------------
Category: Title
Content: U S P S D W
--------------------------------------------------
Category: Title
Content: i
--------------------------------------------------
Category: Title
Content: n e m o W
--------------------------------------------------
Category: Title
Content: l
--------------------------------------------------
Category: Title
Content: I
--------------------------------------------------
Category: Title
Content: h t h g E 5 2 0 2
--------------------------------------------------
Category: Title
Content: i
--------------------------------------------------
Category: Header
Content: 2025 Eighth International Women in Data Science Conference at Prince Sultan University (WiDS PSU)
--------------------------------------------------
Category: Title
Content: Multi-Tiered RAG-Based Cha

In [60]:
# Get the source file path from the metadata
print(documents[0].metadata['source'])

../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf


In [35]:
# Get just the text content of the document
print(documents[0].page_content)

1


In [37]:
# Or loop through all loaded documents (even with sample_size=1)
for doc in documents:
    print("Source:", doc.metadata['source'])
    print("Content:", doc.page_content[:200], "...")  # Print first 200 characters
    print("-" * 50)

Source: ../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf
Content: 1 ...
--------------------------------------------------
Source: ../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf
Content: ©2025 IEEE DOI: 10.1109/WIDS-PSU64963.2025.00041 ...
--------------------------------------------------
Source: ../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf
Content: 4 ...
--------------------------------------------------
Source: ../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf
Content: 0 ...
--------------------------------------------------
Source: ../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf
Content: 0 ...
--------------------------------------------------
Source: ../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf
Content: 0 ...
--------------------------------------------------
Source: ../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf
Content: . ...
---

In [38]:
# Loop through all loaded documents (even if there's only one)
for doc in documents:
    # Access the main text content
    print("--- Page Content ---")
    print(doc.page_content)
    
    # Access the metadata dictionary
    print("\n--- Metadata ---")
    print(f"Source file: {doc.metadata.get('source')}")
    print(f"Page number: {doc.metadata.get('page_number')}")
    print(f"File type: {doc.metadata.get('filetype')}")
    
    # The 'elements' mode returns paragraphs, so 'page_number'
    # might be on the metadata of each element.
    # To be safe, check the 'metadata' of each element.
    
    # You can access any other metadata key, such as
    # "filename", "last_modified", etc.
    print("-" * 50)

--- Page Content ---
1

--- Metadata ---
Source file: ../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf
Page number: 1
File type: application/pdf
--------------------------------------------------
--- Page Content ---
©2025 IEEE DOI: 10.1109/WIDS-PSU64963.2025.00041

--- Metadata ---
Source file: ../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf
Page number: 1
File type: application/pdf
--------------------------------------------------
--- Page Content ---
4

--- Metadata ---
Source file: ../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf
Page number: 1
File type: application/pdf
--------------------------------------------------
--- Page Content ---
0

--- Metadata ---
Source file: ../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf
Page number: 1
File type: application/pdf
--------------------------------------------------
--- Page Content ---
0

--- Metadata ---
Source file: ../Docs/Multi-Tiered_RAG-Based_Chat

In [45]:
loader = DirectoryLoader(
    '../Docs',
    glob="**/*.pdf",
    loader_cls=UnstructuredPDFLoader, # Specifically for PDFs
    loader_kwargs={"languages": ["eng"],
                   "size": {"longest_edge": rounded_edge}, # Set a fixed size if preferred
                   "mode": "elements", 
                   "strategy": "auto",
                   "infer_table_structure": True,
                   "extract_images_in_pdf": False,
                   "show_progress": True,
                   },
    use_multithreading=True,
    max_concurrency=8,
    show_progress=True,
    silent_errors=True,         # skip files that raise errors
    load_hidden=False,          # don't load hidden files/folders
    # recursive=True,             # load files from subfolders recursively
    sample_size=1,            # max 100 files, useful for testing
    randomize_sample=True,      # shuffle files before loading
    sample_seed=1776            # reproducible shuffle for sampling
)

# Actually load the documents
documents = loader.load()

print(f"Total documents loaded: {len(documents)}\n")
# print(documents[0].metadata['source'])  # Display the source of the first document
# print(documents[0])  # Display the content of the first document

direct_pdf = documents.copy()

100%|██████████| 1/1 [00:18<00:00, 18.01s/it]

Total documents loaded: 207



In [46]:
loaded_sources = [doc.metadata.get("source", "") for doc in direct_pdf]
loaded_files = set(loaded_sources)

loaded_files


{'../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf'}

In [47]:
all_keys = set()
for doc in direct_pdf:
    all_keys.update(doc.metadata.keys())
print(f"All unique metadata keys: {all_keys}")

All unique metadata keys: {'file_directory', 'languages', 'source', 'element_id', 'filename', 'last_modified', 'category', 'detection_class_prob', 'parent_id', 'page_number', 'text_as_html', 'coordinates', 'filetype'}


In [48]:
len(direct_pdf)

207

In [49]:
direct_pdf[:1]

[Document(metadata={'source': '../Docs/Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf', 'coordinates': {'points': ((np.float64(30.555555555555554), np.float64(311.40000000000043)), (np.float64(30.555555555555554), np.float64(322.64444444444484)), (np.float64(52.77777777777778), np.float64(322.64444444444484)), (np.float64(52.77777777777778), np.float64(311.40000000000043))), 'system': 'PixelSpace', 'layout_width': 1700, 'layout_height': 2200}, 'last_modified': '2025-09-18T16:01:17', 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 1, 'file_directory': '../Docs', 'filename': 'Multi-Tiered_RAG-Based_Chatbot_for_Mental_Health_Support.pdf', 'category': 'UncategorizedText', 'element_id': 'e520af87dad6ecbfd6bb825cebceccad'}, page_content='1')]

In [50]:
direct_pdf[0].page_content[:]

'1'

In [51]:
# Assuming 'documents' is the list of Document objects you loaded
headings = []

for doc in documents:
    # Get the category from the document's metadata
    category = doc.metadata.get('category')
    
    # Check if the category is a heading or title
    if category in ['Header', 'Title']:
        headings.append(doc)

print(f"Total headings found: {len(headings)}\n")

# Print the content of each found heading
for heading in headings:
    print(f"Category: {heading.metadata.get('category')}")
    print(f"Content: {heading.page_content}")
    print("-" * 50)

Total headings found: 14

Category: Header
Content: 0 0 . 1 3 $ / 5 2 / 2 - 2 9 0 2 - 5 1 3 3 - 8 - 9 7 9 | ) U S P S D i W ( y t i s r e v i n U n a t l u S e c n i r P t a e c n e r e f n o C e c n e i
--------------------------------------------------
Category: Header
Content: e m o W l a n o i t a n r e t n I h t h g i E 5 2 0
--------------------------------------------------
Category: Header
Content: 2025 Eighth International Women in Data Science Conference at Prince Sultan University (WiDS PSU)
--------------------------------------------------
Category: Title
Content: Multi-Tiered RAG-Based Chatbot for Mental Health Support
--------------------------------------------------
Category: Title
Content: II. RELATED WORKS
--------------------------------------------------
Category: Title
Content: C. Conclusion
--------------------------------------------------
Category: Title
Content: III. FRAMEWORK
--------------------------------------------------
Category: Title
Content: A. Why R

This error comes from an invalid color value—usually a resource name like /P0—when a numeric value is needed for gray color in a PDF. The cause is almost always malformed or nonstandard PDFs, and the solution is either updating your tools, regenerating the PDF, or bypassing/repairing problematic resources, depending on your workflow and needs. 

In [ ]:
# import glob
# import os

# loaded_sources = [doc.metadata.get("source", "") for doc in documents]
# all_files = set(glob.glob('../Docs/**/*.pdf', recursive=True))
# loaded_files = set(loaded_sources)
# rejected_files = all_files - loaded_files

# print("Rejected files:")
# print("\n".join(rejected_files))


In [ ]:
# documents[1].metadata.keys()  # List all metadata keys for the second document

In [ ]:
all_keys = set()
for doc in documents:
    all_keys.update(doc.metadata.keys())
print(f"All unique metadata keys: {all_keys}")

#### UnstructuredPDFLoader (152 documents from 152 files)
- Default Behavior: UnstructuredPDFLoader (using the unstructured library) typically aims to process an entire PDF file as one logical document. Even if a PDF has multiple pages, UnstructuredPDFLoader will often concatenate the content of all pages into the page_content of a single LangChain Document object for that PDF file.
- Granularity: It focuses on extracting the overall text and structure of the entire document, rather than splitting it by page. While it might include page_number in the metadata or even in the page_content itself, the primary output for one PDF file is usually one Document object.
- Result: This is why you get 1 document per PDF file. If you have 153 PDF files, you get 153 Document objects.

#### PyMuPDFLoader (966 documents from 153 files)

- Default Behavior: PyMuPDFLoader (using the PyMuPDF library, also known as fitz) typically loads each page of a PDF file as a separate LangChain Document object.
- Granularity: It's designed to provide more granular control at the page level. For a 10-page PDF, PyMuPDFLoader would yield 10 separate Document objects, each representing one page.
- Result: This explains the significant difference. If you divide the total documents by the number of files (966 / 153), you get approximately 6.3 pages per PDF file on average. This is a very common scenario. Each of those 966 documents represents one page from one of your 153 PDF files.

#### Implications for your RAG Pipeline:
- The choice between these loaders (and their output granularity) has significant implications for your RAG pipeline:
    - UnstructuredPDFLoader (Document-level chunks):
        - Pros: Good for maintaining broad context if you want to retrieve large sections or entire small documents at once. Often better at handling complex layouts or scanned documents (especially with hi_res).
        - Cons: If your PDFs are very long, a single Document object might exceed the LLM's context window. You'll definitely need a RecursiveCharacterTextSplitter afterwards to break these large documents into smaller, manageable chunks for your vector store.

#### PyMuPDFLoader (Page-level chunks):
- Pros: Provides immediate, smaller, page-level chunks, which are often a good starting point for RAG. Each chunk comes with clear page metadata. Generally faster for basic text extraction from well-formed PDFs.
- Cons: Might not handle complex layouts, tables, or scanned PDFs as robustly as unstructured. You might still need a RecursiveCharacterTextSplitter if individual pages are too long for your LLM's context window, or if you want to split pages further semantically.

In [ ]:
# How to load multiple PDFs with PyMuPDFLoader
from langchain_community.document_loaders import PyMuPDFLoader

pdf_paths = glob.glob("../Docs/*.pdf")  # list all pdf files

pymu_documents = []
for path in tqdm(pdf_paths, desc="Loading PDFs"):
    loader = PyMuPDFLoader(path)
    docs = loader.load()
    pymu_documents.extend(docs)

print(f"Loaded {len(pymu_documents)} documents from {len(pdf_paths)} files.")
# print(documents[0].metadata['source'])  # Display the source of the first document

pymupdf_loader_docs = documents.copy()

In [ ]:
all_keys = set()
for doc in documents:
    all_keys.update(doc.metadata.keys())
print(f"All unique metadata keys: {all_keys}")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader # Use PyPDFLoader

pdf_files = glob.glob("../Docs/*.pdf")
documents = []

for file_path in tqdm(pdf_files, desc="Loading PDFs"):
    loader = PyPDFLoader(file_path) # No strategy needed for PyPDFLoader
    docs = loader.load()
    documents.extend(docs)

print(f"Loaded {len(documents)} documents from {len(pdf_files)} files using PyPDFLoader.")

pypdf_loader_docs = documents.copy()

In [ ]:
all_keys = set()
for doc in documents:
    all_keys.update(doc.metadata.keys())
print(f"All unique metadata keys: {all_keys}")

# doc.page_content

# from langchain_core.documents.base import Document

# # Create a Document manually
# doc = Documents(
#     page_content="This is a sample page from a PDF.",
#     metadata={"source": "file.pdf", "page": 1}
# )

# print(doc.page_content)  # Output: This is a sample page from a PDF.
# print(doc.metadata)      # Output: {'source': 'file.pdf', 'page': 1}

## Multiple Loader Used

In [ ]:
print(type(direct_loader_docs))
print(type(direct_loader_unstructured_docs))
print(type(direct_loader_unstructured_pdf_docs))
print(type(pymupdf_loader_docs))
print(type(pypdf_loader_docs))

In [ ]:
print(len(direct_loader_docs))
print(len(direct_loader_unstructured_docs))
print(len(direct_loader_unstructured_pdf_docs))
print(len(pymupdf_loader_docs))
print(len(pypdf_loader_docs))

In [ ]:
direct_loader_docs[1].page_content

In [ ]:
direct_loader_unstructured_docs[1].page_content

In [ ]:
direct_loader_unstructured_pdf_docs[1].page_content

In [ ]:
pymupdf_loader_docs[1].page_content

In [ ]:
pypdf_loader_docs[1].page_content

In [ ]:
# pymu_documents is your list from PyMuPDFLoader
if pymupdf_loader_docs:
    print("\n--- Inspecting PyMuPDFLoader Documents ---")
    print(f"First document metadata: {pymupdf_loader_docs[0].metadata}")
    print(f"Second document metadata: {pymupdf_loader_docs[1].metadata}")
    # You should see 'page' or 'page_number' in the metadata, likely starting from 0 or 1.
    # The 'source' will be the same for consecutive pages of the same PDF.

In [ ]:
#  pypdf_loader_docs is your list from PyMuPDFLoader
if pypdf_loader_docs:
    print("\n--- Inspecting PyPDFLoader Documents ---")
    print(f"First document metadata: {pypdf_loader_docs[0].metadata}")
    print(f"Second document metadata: {pypdf_loader_docs[1].metadata}")
    # You should see 'page' or 'page_number' in the metadata, likely starting from 0 or 1.
    # The 'source' will be the same for consecutive pages of the same PDF.

The phrase "all doc loaded in langchain" means you've successfully completed the Document Loading phase of your RAG pipeline. You now have a list of LangChain Document objects (either 153 or 966, depending on whether you used UnstructuredPDFLoader or PyMuPDFLoader for your final load), with their page_content and metadata correctly extracted.

#### What's Next: Text Splitting (Chunking)
- Even if PyMuPDFLoader already split your PDFs into pages, individual pages can still be too large for an LLM's context window. The next crucial step in building your RAG system is Text Splitting (Chunking).

#### Why is Text Splitting Important?
- LLM Context Window Limits: Large language models have a maximum amount of text they can process at once (their "context window"). Your full documents or even single long pages will often exceed this limit.
- Retrieval Relevance: When you perform a retrieval, you want to find the most relevant snippets of information, not necessarily entire documents or pages. Smaller, semantically coherent chunks lead to more precise retrieval.
- Cost and Speed (for API-based LLMs): Smaller inputs mean lower token counts, which can reduce costs and inference time if you were using external LLM APIs (less relevant for purely local LLMs, but still good practice).

#### How to do it:
- You'll typically use a RecursiveCharacterTextSplitter from LangChain. It attempts to split text in a smart way, trying to keep paragraphs and sentences together, and only splitting into smaller units if larger ones exceed your defined chunk_size.

In [ ]:
# print(len(direct_loader_docs))
# print(len(direct_loader_unstructured_docs))
# print(len(direct_loader_unstructured_pdf_docs))
print(len(pymupdf_loader_docs))
print(len(pypdf_loader_docs))

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Assuming 'documents' is your list of loaded LangChain Document objects

# Initialize the text splitter
# chunk_size: The maximum number of characters for each text chunk.
# chunk_overlap: The number of characters to overlap between adjacent chunks.
#                This helps retain context across splits.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # Experiment with this value (e.g., 500, 1500, 2000)
    chunk_overlap=200  # Experiment with this value (e.g., 50, 100, 200)
)

# Split the documents into chunks
chunks_direct_loader_pymupdf = text_splitter.split_documents(pymupdf_loader_docs)

print(f"\nOriginal documents: {len(pymupdf_loader_docs)}")
print(f"Split into {len(chunks_direct_loader_pymupdf)} chunks.")

# You can view a sample chunk as well:
if chunks_direct_loader_pymupdf:
    print("\n--- Sample of the first chunk ---")
    print(f"Content (first 500 characters):\n{chunks_direct_loader_pymupdf[0].page_content[:500]}...")
    print(f"Metadata:\n{chunks_direct_loader_pymupdf[0].metadata}")

The chunks list will now contain many more smaller Document objects, each suitable for embedding and storing in your vector database.

In [ ]:
# Assuming 'documents' is your list of loaded LangChain Document objects

# Initialize the text splitter
# chunk_size: The maximum number of characters for each text chunk.
# chunk_overlap: The number of characters to overlap between adjacent chunks.
#                This helps retain context across splits.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # Experiment with this value (e.g., 500, 1500, 2000)
    chunk_overlap=200  # Experiment with this value (e.g., 50, 100, 200)
)

# Split the documents into chunks
chunks_direct_loader_pypdf = text_splitter.split_documents(pypdf_loader_docs)

print(f"\nOriginal documents: {len(pypdf_loader_docs)}")
print(f"Split into {len(chunks_direct_loader_pypdf)} chunks.")

# You can view a sample chunk as well:
if chunks_direct_loader_pypdf:
    print("\n--- Sample of the first chunk ---")
    print(f"Content (first 500 characters):\n{chunks_direct_loader_pypdf[0].page_content[:500]}...")
    print(f"Metadata:\n{chunks_direct_loader_pypdf[0].metadata}")

#### The next crucial steps in building your local RAG (Retrieval-Augmented Generation) pipeline are:
- Text Embedding: Convert your text chunks into numerical representations (vectors).
- Vector Store/Database: Store these embeddings in a specialized database that allows for efficient similarity search.

#### Text Embedding (Creating Embeddings)
- What it is: Text embedding is the process of converting text (your chunks) into dense numerical vectors. These vectors capture the semantic meaning of the text, such that chunks with similar meanings will have vectors that are "close" to each other in a multi-dimensional space.
- Why it's needed: When a user asks a query, you'll convert that query into an embedding as well. To find relevant documents, you'll then search for document chunks whose embeddings are most similar to the query's embedding.
- Tool: For a local RAG setup, you'll use a local embedding model. HuggingFaceEmbeddings is a common choice as it allows you to load models directly from Hugging Face Hub that can run on your CPU/GPU without external API calls.
- What it is and Why it's needed: Your explanation is spot on. Embeddings are the bridge between human language and machine understanding, enabling semantic search. The "closeness" of vectors in multi-dimensional space directly correlates to the semantic similarity of the original text chunks.
#### Key considerations when using HuggingFaceEmbeddings:
- Model Selection:
    - Performance vs. Size: The Hugging Face MTEB (Massive Text Embedding Benchmark) Leaderboard is your go-to resource for comparing models. Look at "Retrieval Average" scores, but also consider "Model Size" and "Max Tokens." Larger models often offer better performance but require more computational resources (RAM and potentially GPU).
    - Domain Specificity: For highly specialized domains (e.g., medical, legal), fine-tuned models might outperform general-purpose ones.
    - Multilingual Support: If your data isn't exclusively English, ensure the model supports the languages you need.
    - Examples of good local models: Models like BAAI/bge-small-en-v1.5, intfloat/e5-base-v2, and variants of all-MiniLM-L6-v2 are popular choices for their balance of performance and efficiency for local deployment.

#### Hardware Usage (CPU/GPU):
- HuggingFaceEmbeddings typically uses the sentence-transformers library under the hood. This library automatically tries to leverage a GPU if one is available and configured correctly (e.g., PyTorch with CUDA support).
- Explicit Device Setting: You can explicitly set the device to "cpu" or "cuda" (for GPU) when initializing HuggingFaceEmbeddings using model_kwargs={'device': 'cpu'} or model_kwargs={'device': 'cuda'}. This is particularly useful if you have multiple GPUs or want to force CPU usage for testing/resource management.
- Batching: For faster embedding generation, especially on GPUs, process your text chunks in batches. HuggingFaceEmbeddings often has a batch_size parameter.
- Performance Monitoring: If you have a GPU, monitor its utilization (e.g., with nvidia-smi on Linux) to ensure it's being used effectively. Sometimes, CPU bottlenecks (e.g., slow data loading) can lead to low GPU utilization.
- Installation: You'll need to install the sentence-transformers package: pip install sentence-transformers. If you plan to use a GPU, ensure your PyTorch installation supports CUDA.
- Normalization: For cosine similarity (a common metric for vector similarity search), it's often recommended to normalize the embeddings. Some models produce normalized embeddings by default, while others might require you to set normalize_embeddings=True in encode_kwargs when initializing HuggingFaceEmbeddings.

#### Vector Store/Database
- What it is and Why it's needed: Once you have your embeddings, you need a place to store them that allows for fast "nearest neighbor" searches. This is where vector databases shine. They are optimized for efficiently finding vectors that are semantically close to a given query vector.

#### Next steps for the Vector Store:
- Choice of Vector Store: For local RAG, popular choices include:
    - ChromaDB: A lightweight, easy-to-use vector database that can run entirely locally without complex setup. Excellent for getting started.
    - FAISS (Facebook AI Similarity Search): A library for efficient similarity search and clustering of dense vectors. It's an in-memory index, meaning it's very fast but doesn't persist data between runs unless you explicitly save and load the index. Good for smaller datasets or if you manage persistence yourself.
    - Milvus Lite/Qdrant Lite: Lighter versions of their full-fledged vector database counterparts, offering more features than FAISS while still being suitable for local deployments.
- Indexing: Once you choose a vector store, you'll need to "index" your document embeddings into it. This process organizes the vectors for efficient search.
- Similarity Search: When a user query comes in, you'll convert it to an embedding using the same HuggingFaceEmbeddings model. Then, you'll perform a similarity search in your vector store to retrieve the most relevant document chunks. Common similarity metrics include cosine similarity, dot product, or Euclidean distance.
- By focusing on these two steps – robust text embedding and efficient vector storage – you'll build the core retrieval component of your local RAG pipeline.


In [ ]:
len(chunks_direct_loader_pymupdf), len(chunks_direct_loader_pypdf)

In [ ]:
def two_sum(nums, target):
    num_to_index = {}
    for i, num in enumerate(nums):
        complement = target - num
        if complement in num_to_index:
            return [num_to_index[complement], i]
        num_to_index[num] = i

In [ ]:
nums = [3, 7, 11, 15, 2]
target = 17
result = two_sum(nums, target)
print(result)  # Output: [0, 1]

In [ ]:
num_to_index = {}
for i, num in enumerate(nums):
    print(target - num)
    complement = target - num
    print(f":::::Current num_to_index: {num_to_index}")
    if complement in num_to_index:
        print(f"Found: {num_to_index[complement]}, {i}")

    # print(f"Current num_to_index: {num_to_index}")
    num_to_index[num] = i